# 01 - Waste scene classifier (AerialWaste)

**Question the model answers:** for one 210 m x 210 m piece of overhead imagery, *is there a waste dump here?*
and *which kinds of waste are visible?* - with **Plastic** as the first and most important type.

* Data: AerialWaste v3 (Politecnico di Milano / ARPA Lombardia), 11,703 images labelled by professional photo
  interpreters, official train / test split. Validation is carved out of training; **the test split is scored once**.
* Model: ConvNeXt-Tiny (ImageNet-22k pretrained) + two heads (waste, waste types). Unknown type labels are masked.
* Classic baseline: hand-crafted colour/edge/texture features + logistic regression, and a "white plastic" pixel rule.
* Runtime: Colab T4. First run ~30 min download + ~1.5 h training; later runs reuse the Drive copy.

In [ ]:
# Setup: GPU, Google Drive (click "Connect to Google Drive" when asked), code
import os, sys, subprocess, json, pathlib
import shutil as _sh
if _sh.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
from google.colab import drive
drive.mount("/content/drive")
RUNS = pathlib.Path("/content/drive/MyDrive/waste_runs"); RUNS.mkdir(parents=True, exist_ok=True)
REPO = "/content/waste_detection"
if not os.path.exists(REPO):
    !git clone -q https://github.com/inayatarshad/waste_detection.git {REPO}
else:
    !git -C {REPO} pull -q
!pip install -q -r {REPO}/requirements.txt
sys.path.insert(0, f"{REPO}/src")
import numpy as np, torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)
if DEVICE == "cpu":
    print("NOTE: no GPU in this session. Testing an existing model works on CPU (slower; "
          "test-time augmentation is switched off). Training needs a GPU.")

In [ ]:
# AerialWaste: download once from Zenodo (18.8 GB), resize to 640 px, keep a 1-2 GB copy on Drive.
# Later sessions only unpack the Drive copy (a few minutes).
import hashlib, requests, tarfile, zipfile, shutil
from tqdm.auto import tqdm
from wastedet import aerialwaste as AW

META = RUNS / "aerialwaste_meta"; META.mkdir(exist_ok=True)
CACHE = pathlib.Path("/content/aw640")                 # fast local disk while training
CACHE_TAR = RUNS / "aerialwaste_640.tar"               # persistent copy on Drive
RAW = pathlib.Path("/content/aw_raw")

def download(url, dst, md5=None):
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        h = hashlib.md5()
        with open(dst, "wb") as f:
            for chunk in r.iter_content(1 << 22):
                f.write(chunk); h.update(chunk)
    if md5 and h.hexdigest() != md5:
        raise RuntimeError(f"checksum mismatch for {dst}")

rec = requests.get("https://zenodo.org/api/records/12607190", timeout=60).json()
files = {f["key"]: f for f in rec["files"]}
print("Zenodo files:", sorted(files))
for key, f in files.items():
    if key.endswith(".json") and not (META / key).exists():
        download(f["links"]["self"], META / key, f["checksum"].split(":")[-1])
TRAIN_JSON = next(META / k for k in files if k.endswith(".json") and "train" in k.lower())
TEST_JSON = next(META / k for k in files if k.endswith(".json") and "test" in k.lower())

names = [r.file_name for r in AW.load_records(TRAIN_JSON) + AW.load_records(TEST_JSON)]
if not CACHE.exists() and CACHE_TAR.exists():
    with tarfile.open(CACHE_TAR) as t:
        t.extractall("/content")
have = sum((CACHE / n).exists() for n in names)
if have < len(names):
    for key, f in files.items():
        if not key.endswith(".zip"):
            continue
        RAW.mkdir(exist_ok=True); zpath = pathlib.Path("/content") / key
        print("downloading", key, f["size"] / 1e9, "GB"); download(f["links"]["self"], zpath, f["checksum"].split(":")[-1])
        with zipfile.ZipFile(zpath) as z:
            z.extractall(RAW)
        zpath.unlink()
        n = AW.prepare_cache(AW.index_images(RAW), CACHE, names, size=640)
        shutil.rmtree(RAW)                              # keep the local disk free
        print(f"cached {n}/{len(names)} images")
    with tarfile.open(CACHE_TAR, "w") as t:
        t.add(CACHE, arcname=CACHE.name)
have = sum((CACHE / n).exists() for n in names)
print(f"{have} of {len(names)} images ready at 640 px")
assert have == len(names), "some images are missing - re-run this cell"

In [ ]:
# Labels, splits and leakage checks
from wastedet.imageio_utils import load_rgb, dhash
TYPES = AW.DEFAULT_TYPES
train_all = AW.load_records(TRAIN_JSON, TYPES); test_recs = AW.load_records(TEST_JSON, TYPES)
train_recs, val_recs = AW.stratified_split(train_all, val_frac=0.15, seed=0)
for name, rs in (("train", train_recs), ("val", val_recs), ("test", test_recs)):
    print(name, json.dumps(AW.summarise(rs, TYPES)))
assert not ({r.file_name for r in train_all} & {r.file_name for r in test_recs}), "file in both train and test"

# near-duplicate images across train and test (same place re-used) - reported, the official split is kept
# (computed once and saved to Drive; later runs read the saved number)
SUMMARY = RUNS / "classifier_v1" / "data_summary.json"
if SUMMARY.exists() and "near_duplicates_test_in_train" in json.loads(SUMMARY.read_text()):
    near = json.loads(SUMMARY.read_text())["near_duplicates_test_in_train"]
else:
    def hashes(rs):
        return np.array([dhash(load_rgb(CACHE / r.file_name)) for r in tqdm(rs, desc="hash")], dtype=np.uint64)
    h_tr, h_te = hashes(train_all), hashes(test_recs)
    bits = lambda v: np.unpackbits(v.view(np.uint8).reshape(-1, 8), axis=1).sum(1)
    near = sum(int((bits(h_tr ^ h) <= 3).any()) for h in h_te)
    (RUNS / "classifier_v1").mkdir(exist_ok=True)
    SUMMARY.write_text(json.dumps(
        {k: AW.summarise(v, TYPES) for k, v in (("train", train_recs), ("val", val_recs), ("test", test_recs))}
        | {"near_duplicates_test_in_train": near}, indent=1))
print(f"test images with a near-duplicate in training: {near} of {len(test_recs)}")

In [ ]:
# Classic baseline (no deep learning), same images. Threshold on validation, test once.
from wastedet import metrics as M
from wastedet.baselines import HandcraftedBaseline, handcrafted_features, WHITE_COL
FEATS = RUNS / "classifier_v1" / "baseline_features.npz"
if FEATS.exists():
    F = dict(np.load(FEATS))
else:
    f = lambda rs: np.stack([handcrafted_features(load_rgb(CACHE / r.file_name)) for r in tqdm(rs)])
    F = {"tr": f(train_recs), "va": f(val_recs), "te": f(test_recs)}; np.savez(FEATS, **F)
y = lambda rs: np.array([r.waste for r in rs])
base = HandcraftedBaseline().fit(F["tr"], y(train_recs))
b_thr = M.best_threshold(y(val_recs), base.score(F["va"]))["threshold"]
b_scores = base.score(F["te"])
baseline_rep = {"waste": {"ap": M.average_precision(y(test_recs), b_scores), **M.binary_at(y(test_recs), b_scores, b_thr)}}
# "white plastic" rule for the plastic type: share of bright, unsaturated pixels
pm = lambda rs: np.array([AW.type_mask_of(r)[0] > 0 for r in rs])
pl = lambda rs: np.array([r.types[0] for r in rs])
w_thr = M.best_threshold(pl(val_recs)[pm(val_recs)], F["va"][pm(val_recs), WHITE_COL], grid=np.linspace(0, 0.5, 101))["threshold"]
k = pm(test_recs)
baseline_rep["plastic_white_rule"] = {"ap": M.average_precision(pl(test_recs)[k], F["te"][k, WHITE_COL]),
                                      **M.binary_at(pl(test_recs)[k], F["te"][k, WHITE_COL], w_thr)}
print(json.dumps(baseline_rep, indent=1))

In [ ]:
# Train the classifier (best epoch and thresholds chosen on validation).
# Safe to re-run after a disconnect: it continues from the last finished epoch (last.pt on Drive),
# and a finished run is not trained again.
# USE_EXISTING_BEST = True skips training and uses the best.pt already on Drive
# (e.g. a run that stopped before finishing).
from wastedet.train_cls import ClsConfig, train_classifier, finalize_best
USE_EXISTING_BEST = False
cfg = ClsConfig(types=TYPES, size=640, epochs=20, batch_size=16, lr=2e-4, num_workers=2,
                tta=(DEVICE == "cuda"))   # TTA = 4 flipped/rotated views; too slow on CPU
train_ds = AW.WasteSceneDataset(train_recs, CACHE, cfg.size, train=True)
val_ds = AW.WasteSceneDataset(val_recs, CACHE, cfg.size)
test_ds = AW.WasteSceneDataset(test_recs, CACHE, cfg.size)
OUT = RUNS / "classifier_v1"
if USE_EXISTING_BEST:
    res = finalize_best(cfg, val_ds, OUT, DEVICE)
else:
    _last = OUT / "last.pt"
    _done = _last.exists() and torch.load(_last, map_location="cpu", weights_only=False)["epoch"] >= cfg.epochs
    assert DEVICE == "cuda" or _done, "Training needs a GPU: Runtime > Change runtime type > T4 GPU (or set USE_EXISTING_BEST = True)"
    res = train_classifier(cfg, train_ds, val_ds, OUT)   # a finished run is not trained again

In [ ]:
# Score the test split ONCE
from wastedet.train_cls import load_classifier, score_test_split
model, ckpt = load_classifier(OUT / "best.pt", DEVICE)
rep, tp = score_test_split(model, test_ds, ckpt["thresholds"], TYPES, OUT / "test_report.json", cfg, device=DEVICE,
                           extra={"classic_baseline": baseline_rep, "best_epoch": ckpt["best_epoch"],
                                  "device": DEVICE, "tta": cfg.tta})
w = rep["waste"]
print(f"WASTE   AP {w['ap']:.3f} | precision {w['precision']:.3f} recall {w['recall']:.3f} F1 {w['f1']:.3f} "
      f"| false alarms {w['false_alarm_rate']:.3f}")
print(f"classic AP {baseline_rep['waste']['ap']:.3f} | F1 {baseline_rep['waste']['f1']:.3f}")
for n, t in rep["types"].items():
    print(f"{n:35s} AP {t['ap']:.3f} (chance {t['chance_ap']:.3f}) | within waste AP {t['ap_within_waste']:.3f} "
          f"(chance {t['chance_ap_within_waste']:.3f}) | {t['positives']} positives")
print("by source:", json.dumps(rep.get("by_source", {}), indent=1))

In [ ]:
# Figures for the report / slides
from wastedet import plots
FIG = OUT / "figures"; FIG.mkdir(exist_ok=True)
if (OUT / "history.json").exists():
    plots.plot_history(json.loads((OUT / "history.json").read_text()),
                   FIG / "training_curves.png", keys=("val_ap", "val_types_map", "val_plastic_ap"),
                   title="Waste classifier - training")
plots.plot_pr({"ConvNeXt-T (ours)": (tp["y_waste"], tp["s_waste"], rep["waste"]["ap"]),
               "Classic features + LR": (tp["y_waste"], b_scores, baseline_rep["waste"]["ap"])},
              FIG / "pr_waste.png", ref_point=(0.7954, 0.8189, "Published ResNet50-FPN (dataset v1)"))
plots.plot_type_aps(rep, FIG / "type_ap.png")

# examples: correct finds, false alarms, misses (with the model's attention map)
thr = ckpt["thresholds"]["waste"]; sw, yw = tp["s_waste"], tp["y_waste"]
pick = lambda m, n=4: [i for i in np.argsort(-sw * (1 if n > 0 else -1)) if m[i]][:abs(n)]
groups = [("found", pick((yw == 1) & (sw >= thr))), ("false alarm", pick((yw == 0) & (sw >= thr))),
          ("missed", pick((yw == 1) & (sw < thr), -4))]
imgs, cams, titles = [], [], []
for label, idx in groups:
    for i in idx:
        im = test_ds.image(i); x = test_ds[i][0][None].to(DEVICE)
        imgs.append(im); cams.append(model.cam(x)[0].cpu().numpy())
        titles.append(f"{label}: score {sw[i]:.2f}, plastic {tp['s_types'][i, 0]:.2f}")
plots.grid_with_cam(imgs, cams, titles, FIG / "examples_cam.png")
from IPython.display import Image as IImage, display
for f in ("training_curves.png", "pr_waste.png", "type_ap.png", "examples_cam.png"):
    if (FIG / f).exists():
        display(IImage(str(FIG / f)))

In [ ]:
# ONNX export (checked against PyTorch) + results zip
from wastedet.models import export_classifier_onnx
rel = export_classifier_onnx(model.float().cpu(), OUT / "classifier.onnx", size=640)
print(f"ONNX max relative difference {rel:.1e}")
import shutil
shutil.make_archive(str(RUNS / "results_classifier_v1"), "zip", OUT)
print("saved", RUNS / "results_classifier_v1.zip")